多头自注意力（MHA）前向过程。

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

def mha_forward(x, W_q, W_k, W_v, W_o, num_heads, mask=None):
    B, T, D = x.size()
    d_head = D // num_heads
    q = x @ W_q
    k = x @ W_k
    v = x @ W_v
    q_head = q.view(B, T, num_heads, d_head).transpose(1, 2) # (B, num_heads, T, d_head)
    k_head = k.view(B, T, num_heads, d_head).transpose(1, 2)
    v_head = v.view(B, T, num_heads, d_head).transpose(1, 2)
    attn = q_head @ k_head.transpose(-2, -1) / math.sqrt(d_head) # (B, num_heads, T, T)
    if mask is not None:
        attn = attn.masked_fill(mask == 0, float('-inf'))
    attn = F.softmax(attn, dim=-1)
    out_head = attn @ v_head # (B, num_heads, T, d_head)
    out = out_head.transpose(1, 2).contiguous().view(B, T, D) @ W_o
    return out, attn

此处 `contiguous()` 的作用是，重新分配内存，把数据按当前逻辑顺序拷贝成真正连续的布局。

In [4]:
import torch
x = torch.randn(2, 3)      # 内存连续
y = x.transpose(0, 1)      # 只改了 stride，没动数据！

print(x.is_contiguous())   # True
print(y.is_contiguous())   # False

y_c = y.contiguous()
print(y_c.is_contiguous())  # True

True
False
True


mask 有两种常见形式：
1. Padding Mask — 屏蔽填充位（pad token），形状 (B, 1, 1, T)；
2. Causal Mask（因果掩码） — 解码器自回归，只看左边，形状 (1, 1, T, T) 上三角为 0。

In [21]:
import torch

def make_padding_mask(seq, pad_idx=0):
    valid = (seq != pad_idx)
    # 外积：query维度列向量 × key维度行向量 → (B, T, T)
    # valid[:, :, None] = (B, T, 1)
    # valid[:, None, :] = (B, 1, T)
    mask = valid.unsqueeze(2) & valid.unsqueeze(1)  # (B, T, T)
    return mask.unsqueeze(1)  # (B, 1, T, T)

def make_causal_mask(T):
    mask = torch.tril(torch.ones(T, T)).bool()
    return mask.unsqueeze(0).unsqueeze(0)

def make_decoder_mask(seq, pad_idx=0):
    B, T = seq.shape
    pad_mask = make_padding_mask(seq, pad_idx)
    causal_mask = make_causal_mask(T)
    return pad_mask & causal_mask

seq = torch.tensor([
    [3, 5, 7, 0, 0],   # 后两个是 pad
    [1, 2, 3, 4, 0],   # 最后一个是 pad
])
make_decoder_mask(seq)

tensor([[[[ True, False, False, False, False],
          [ True,  True, False, False, False],
          [ True,  True,  True, False, False],
          [False, False, False, False, False],
          [False, False, False, False, False]]],


        [[[ True, False, False, False, False],
          [ True,  True, False, False, False],
          [ True,  True,  True, False, False],
          [ True,  True,  True,  True, False],
          [False, False, False, False, False]]]])